In [1]:
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [3]:
fusion = pd.read_csv(
    "../dataset/processed/adaptive_fusion_dataset.csv"
)

X = fusion.drop(columns=["Time_taken (min)"])

y = fusion["Time_taken (min)"]
fusion.head()

,Traffic_Score,Workload,Multiple_Deliveries,Peak,Festival,Rider_Experience,Ratings,Traffic_Workload,Demand_Index,Rider_Load,...,Experience,Delivery_Index,Weather_Impact,Experience_Index,Order_Hour,Pickup_Hour,Weekend,Month,Delivery_person_Age,Time_taken (min)
0,4,3.0,3.0,3,0,151.2,4.2,12.0,16,37.80,...,151.2,41.122328,41.122328,30.240000,21,22,1,2,36.0,46
1,3,1.0,1.0,2,0,98.7,4.7,3.0,9,49.35,...,98.7,18.726956,31.211593,24.675000,14,15,1,2,21.0,23
2,2,1.0,1.0,0,0,108.1,4.7,2.0,2,54.05,...,108.1,27.575720,82.727161,36.033333,17,17,0,3,23.0,21
3,1,0.0,0.0,1,0,146.2,4.3,0.0,2,146.20,...,146.2,2.930258,17.581547,73.100000,9,9,1,2,34.0,20
4,4,1.0,1.0,3,0,112.8,4.7,4.0,16,56.40,...,112.8,77.586473,77.586473,22.560000,19,20,0,2,24.0,41


In [4]:
eta_model = joblib.load(
    "../Modelv3/adaptive_eta_engine.pkl"
)

In [5]:
X_train, X_cal, y_train, y_cal = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [8]:
cal_predictions = eta_model.predict(X_cal)
cal_predictions

array([30.961443, 35.21694 , 32.612194, ..., 22.264189, 28.645802,
       36.091278], dtype=float32)

In [9]:
residuals = np.abs(y_cal - cal_predictions)

print(residuals.describe())

count    9099.000000
mean        3.131990
std         2.393152
min         0.000267
25%         1.302998
50%         2.685711
75%         4.387320
max        25.748281
Name: Time_taken (min), dtype: float64


In [10]:
alpha = 0.05

q = np.quantile(
    residuals,
    1 - alpha
)

print(f"95% Residual Quantile: {q:.2f} minutes")

95% Residual Quantile: 7.66 minutes


In [11]:
sample = X.iloc[[0]]

eta = eta_model.predict(sample)[0]

lower = eta - q

upper = eta + q

print(f"Predicted ETA : {eta:.2f} min")
print(f"95% CI        : ({lower:.2f}, {upper:.2f})")

Predicted ETA : 45.94 min
95% CI        : (38.28, 53.59)


In [12]:
lower = cal_predictions - q
upper = cal_predictions + q

coverage = np.mean(
    (y_cal >= lower) &
    (y_cal <= upper)
)

print(f"Coverage: {coverage:.3f}")

Coverage: 0.950


In [13]:
interval_model = {
    "eta_model": eta_model,
    "quantile": q,
    "alpha": alpha
}

joblib.dump(
    interval_model,
    "../Modelv3/eta_confidence_interval.pkl"
)

['../Modelv3/eta_confidence_interval.pkl']